# **Using RecursiveTextSPlitters for Chunking**

In [24]:
# Importing Libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import pickle
import os

In [25]:
#importing the processed data
with open('../data/processed/playbook_documents.pkl','rb') as f:
    playbook_documents = pickle.load(f)
print(f"Loaded {len(playbook_documents)} log documents from pickle")

Loaded 174 log documents from pickle


In [28]:
#importing the processed data
with open('../data/processed/log_documents.pkl', 'rb') as f:
    log_documents = pickle.load(f)
print(f"Loaded {len(log_documents)} log documents from pickle")


Loaded 4120 log documents from pickle


In [13]:
# Test on a small sample first
test_docs = log_documents[:100]  # Only 100 documents

for chunk_size in [500, 1000, 1500]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=200
    )
    chunks = splitter.split_documents(test_docs)
    print(f"chunk_size={chunk_size}: {len(chunks)} chunks from 100 documents")

chunk_size=500: 493 chunks from 100 documents
chunk_size=1000: 252 chunks from 100 documents
chunk_size=1500: 176 chunks from 100 documents


In [14]:
print(chunks[1])

page_content='USER-1555-WinLogBeat ::: {"@metadata":{"beat":"winlogbeat","type":"_doc","version":"8.2.2"},"@timestamp":"2024-07-26T11:48:46.952Z","USER-0015-0217":{"ephemeral_id":"47434381-3d10-4395-bc1c-5cdb52ecb35e","id":"12e13a9a-252a-493f-9c35-fa4d61abd0bc","name":"USER-0015-0040","type":"winlogbeat","version":"8.2.2"},"ecs":{"version":"8.0.0"},"event":{"action":"Directory ORG-0053 Access","code":"4662","created":"2024-07-26T11:48:47.739Z","kind":"event","outcome":"success","provider":"Microsoft-ORG-0362-Security-Auditing"},"USER-0015":{"name":"USER-0015-0034.example.internal"},"log":{"level":"information"},"message":"An operation was performed on an ORG-0181.\n\nSubject :\n\tSecurity ID:\t\tS-1-5-21-1000000000-2000000000-3000000000-0227\n\tAccount Name:\t\tUSER-0263\n\tAccount Domain:\t\tMDS-ORG-0007\USER-0005\tORG-0063 ID:\t\t0xB2CFBC6D\n\nORG-0181:\n\tORG-0181 Server:\t\tDS\n\tORG-0181 Type:\t\t%{bf967aa5-0de6-11d0-a285-00aa003049e2}\n\tORG-0181 Name:\t\t%{aa83d7dc-4e46-446b-baa

In [15]:
test_docs

[Document(metadata={'timestamp': '1721994830.9275465', 'src_ip': '', 'dst_ip': '', 'username': 'USER-1691', 'severity': '', 'label_binary': 'benign', 'src_port': '', 'dst_port': '', 'protocol': '', 'attack_tactics': '[]'}, page_content="<Event xmlns='http://schemas.microsoft.com/win/2004/08/events/event'><ORG-0111><Provider Name='Microsoft-ORG-0362-Security-Auditing' Guid='{54849625-5478-4994-a5ba-3e3b0328c30d}'/><EventID>4634</EventID><Version>0</Version><Level>0</Level><Task>12545</Task><Opcode>0</Opcode><Keywords>0x8020000000000000</Keywords><TimeCreated ORG-0111Time='2024-07-26T11:45:50.407036700Z'/><EventRecordID>48381480</EventRecordID><Correlation/><Execution ProcessID='656' ThreadID='5072'/><Channel>Security</Channel><Computer>USER-0015-1171.example.internal</Computer><Security/></ORG-0111><EventData><Data Name='TargetUserSid'>S-1-5-18</Data><Data Name='TargetUserName'>IP-6440022A$</Data><Data Name='TargetDomainName'>ORG-0406</Data><Data Name='TargetORG-0063Id'>0x3ac37122</Data

In [ ]:
#reaffirming the log_documents remains 5000
print(f"Length of new log_documents: {len(log_documents)}")

Length of new log_documents: 4120


In [30]:
#Chunking the log documents
splitter= RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n",".", ",", " ", ""],
    chunk_size= 1000,
    chunk_overlap = 200,
)
log_chunks = splitter.split_documents(log_documents)
print(f"Loaded {len(log_chunks)} the chunks from log documents")
print(f"first_chunk: {log_chunks[0].page_content[:100]}")
print(f"Meta_data: {log_chunks[0].metadata} meta data of chunked documents")

Loaded 11016 the chunks from log documents
first_chunk: <Event xmlns='http://schemas.microsoft.com/win/2004/08/events/event'><ORG-0111><Provider Name='Micro
Meta_data: {'timestamp': '1721994830.9275465', 'src_ip': '', 'dst_ip': '', 'username': 'USER-1691', 'severity': '', 'label_binary': 'benign', 'src_port': '', 'dst_port': '', 'protocol': '', 'attack_tactics': '[]'} meta data of chunked documents


In [31]:
# Viewing multiple chunks 
print("\nMetadata for first 3 chunks:")
for i in range(min(3, len(log_chunks))):
    print(f"Chunk {i+1}: {log_chunks[i].metadata}")


Metadata for first 3 chunks:
Chunk 1: {'timestamp': '1721994830.9275465', 'src_ip': '', 'dst_ip': '', 'username': 'USER-1691', 'severity': '', 'label_binary': 'benign', 'src_port': '', 'dst_port': '', 'protocol': '', 'attack_tactics': '[]'}
Chunk 2: {'timestamp': '1721994830.9275465', 'src_ip': '', 'dst_ip': '', 'username': 'USER-1691', 'severity': '', 'label_binary': 'benign', 'src_port': '', 'dst_port': '', 'protocol': '', 'attack_tactics': '[]'}
Chunk 3: {'timestamp': '1721994557.2679317', 'src_ip': '', 'dst_ip': '', 'username': '', 'severity': 'Info', 'label_binary': 'benign', 'src_port': '', 'dst_port': '', 'protocol': '', 'attack_tactics': '[]'}


### **Chunking the Playbook Processed Dataset**

In [18]:
#Chunking the log documents
splitter= RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n",".", ",", " ", ""],
    chunk_size= 1000,
    chunk_overlap = 200,
)
playbook_chunks = splitter.split_documents(playbook_documents)
print(f"Loaded {len(playbook_chunks)} the chunks from log documents")
print(f"first_chunk: {playbook_chunks[0].page_content[:100]}")
print(f"Meta_data: {playbook_chunks[0].metadata} meta data of chunked documents")




Loaded 174 the chunks from log documents
first_chunk: Phase Identification: Triage alert, confirm IOC via EDR, snapshot affected host
Phase Containment: I
Meta_data: {'incident_id': 'IR-2025-0012', 'incident_type': 'Ransomware', 'severity': 'High', 'final_status': 'Resolved'} meta data of chunked documents


In [19]:
# Viewing multiple chunks 
print("\nMetadata for first 3 chunks:")
for i in range(min(3, len(playbook_chunks))):
    print(f"Chunk {i+1}: {playbook_chunks[i].metadata}")


Metadata for first 3 chunks:
Chunk 1: {'incident_id': 'IR-2025-0012', 'incident_type': 'Ransomware', 'severity': 'High', 'final_status': 'Resolved'}
Chunk 2: {'incident_id': 'IR-2025-0013', 'incident_type': 'Data Breach', 'severity': 'Critical', 'final_status': 'Resolved'}
Chunk 3: {'incident_id': 'IR-2025-0014', 'incident_type': 'Credential Dumping', 'severity': 'Medium', 'final_status': 'Resolved'}


### Saving the Chunks from Playbooks and Logs 

In [21]:
#Creating a new directory for the chunks 
os.makedirs('./data/procesed_chunks', exist_ok = True)

In [22]:
#Saving the Playbook_chunks 
with open('../data/processed/playbook_chunks.pkl', 'wb') as f:
    pickle.dump(playbook_chunks, f)

In [32]:
#Saving the log_chunks
with open('../data/processed/log_chunks.pkl', 'wb') as f:
    pickle.dump(log_chunks, f)